# AdaptAuth-ML: demostración de ejecución

**Autenticación adaptativa basada en Machine Learning para la detección de accesos anómalos en PYMES del sector de servicios empresariales, Lima 2026.**

Tesista: Ashly Herminia Jasmín Chávez Marón · Asesor: Ing. Daniel Alfonso Quinto Pazce · UNMSM, Tesis II

Código fuente: [github.com/AshlyChavez/adaptauth-ml](https://github.com/AshlyChavez/adaptauth-ml)

| Paso | Qué se ejecuta | Tiempo aproximado |
|---|---|---|
| 1 | Descarga del código desde GitHub | segundos |
| 2 | Descarga del dataset SuspiciousLogin | segundos |
| 3 | Entrenamiento y evaluación: reglas estáticas, Isolation Forest y AdaptAuth-ML | 1 a 2 min |
| 4 | Verificación de requisitos no funcionales y explicación de alertas | 2 a 4 min |
| 5 | CERT r4.2: historial del empleado, validación cruzada por empleado | varios minutos |

Para ejecutar todo: **Entorno de ejecución → Ejecutar todo**.

## Paso 1. Descarga del código desde GitHub

In [ ]:
import os
os.chdir("/content")
!rm -rf adaptauth-ml
!git clone -q https://github.com/AshlyChavez/adaptauth-ml.git
os.chdir("/content/adaptauth-ml")
!pip -q install shap
print("Estructura del repositorio:")
!ls

## Paso 2. Dataset SuspiciousLogin (Google Workspace)

Inicios de sesión exitosos de dos instituciones durante ocho meses. La etiqueta es la marca de sospecha de Google Workspace.

In [ ]:
import glob
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 200)

if not os.path.exists("/content/SL"):
    !git clone -q --depth 1 https://github.com/salomaopena/SuspiciousLoginDataset.git /content/SL
SL = glob.glob("/content/SL/**/suspicious_logins_public_v1.csv", recursive=True)[0]
d = pd.read_csv(SL)
print(f"Eventos: {len(d):,} | Usuarios: {d['actor_pseudo_id'].nunique():,} | Sospechosos: {d['label'].mean()*100:.2f} %")
display(d[["event_hour", "ip_country", "new_ip", "is_new_country", "impossible_travel", "distinct_ips_30d", "logins_30d", "label"]].head())

## Paso 3. Entrenamiento y evaluación

Se comparan tres mecanismos sobre los mismos datos de prueba:

- **Reglas estáticas** (línea base): IP nueva, país nuevo, viaje imposible, hora atípica.
- **Isolation Forest** aislado.
- **AdaptAuth-ML**: Isolation Forest + clasificador supervisado, con umbrales θ1 (OTP) y θ2 (bloqueo).

Partición temporal: entrenamiento de diciembre a marzo, calibración en abril y prueba en mayo y junio.

In [ ]:
!python run_suspiciouslogin.py --data "{SL}"

### Resultados en el conjunto de prueba (meta: F1 ≥ 0.60)

In [ ]:
t = pd.read_csv("results_sl/tabla_comparativa_sl.csv")
cols = ["modelo", "auc_roc", "auc_pr", "f1_theta1", "precision_theta1", "recall_theta1", "fpr_theta1"]
display(t[[c for c in cols if c in t.columns]].round(3))
for f in ["comparacion_modelos_sl", "distribucion_riesgo_sl", "importancia_grupos_sl", "f1_por_grupo_sl"]:
    ruta = f"results_sl/figuras/{f}.png"
    if os.path.exists(ruta):
        display(Image(ruta, width=850))

## Paso 4. Verificación de requisitos no funcionales

Se verifica el tiempo de respuesta, el score de accesos legítimos, el reentrenamiento seguro, la contingencia OTP cuando el motor falla, el registro de eventos y la explicación de cada alerta con sus tres variables más influyentes (SHAP).

In [ ]:
!python run_requisitos_sl.py --data "{SL}"

In [ ]:
display(Markdown("### Matriz de cumplimiento"))
display(pd.read_csv("results_requisitos_sl/matriz_cumplimiento_requisitos.csv"))
display(Markdown("### Distribución de decisiones en prueba"))
reg = pd.read_csv("results_requisitos_sl/registro_eventos_prueba.csv")
display(reg["decision"].value_counts().to_frame("eventos"))
display(Markdown("### Ejemplos de alertas explicadas (top-3 variables)"))
display(pd.read_csv("results_requisitos_sl/explicacion_alertas_top3.csv").sort_values("score_R", ascending=False).head(8))

## Paso 5. CERT Insider Threat r4.2: historial del empleado

Organización de 1 000 empleados (Carnegie Mellon University). Validación cruzada de 5 grupos de empleados: ningún empleado aparece en entrenamiento y prueba a la vez.

Los datos se leen desde **Mi unidad → AdaptAuth-ML → CERT_r4.2**. Si `EJECUTAR_CERT = False`, se muestran los resultados ya guardados en el repositorio.

In [ ]:
EJECUTAR_CERT = True

CERT = "/content/drive/MyDrive/AdaptAuth-ML/CERT_r4.2"
if EJECUTAR_CERT:
    from google.colab import drive
    drive.mount("/content/drive")
    if glob.glob(f"{CERT}/**/logon.csv", recursive=True):
        !python run_cert_v2.py --dir "{CERT}"
    else:
        print("No se encontró CERT_r4.2 en Drive; se muestran los resultados guardados en el repositorio.")

In [ ]:
display(Markdown("### Resultados CERT r4.2 (meta: AUC ≥ 0.90; recall ≥ 0.70 con FPR ≤ 0.1 %)"))
c = pd.read_csv("results_cert_v2/tabla_comparativa_cert_v2.csv")
display(c[["modelo", "auc_roc", "auc_pr", "f1_theta1", "recall_theta1", "fpr_theta1"]].round(4))
display(Markdown("### Recall según el presupuesto de falsas alarmas (AdaptAuth-ML)"))
display(pd.read_csv("results_cert_v2/puntos_operacion_cert_v2.csv")[["fpr_prueba", "recall", "precision", "alertas_falsas", "dias_detectados"]].round(4))
display(Markdown("### Resultados por unidad funcional"))
display(pd.read_csv("results_cert_v2/por_unidad_cert_v2.csv").round(3))

## Resumen

| Dataset | Resultado principal | Meta |
|---|---|---|
| SuspiciousLogin | F1 0.628 · precisión 0.851 · FPR 0.71 % | F1 ≥ 0.60 ✔ |
| CERT r4.2 | AUC-ROC 0.947 · recall 0.70 con FPR 0.11 % | AUC ≥ 0.90 ✔ · recall en el límite |

AdaptAuth-ML supera de forma significativa a las reglas estáticas en ambos datasets (prueba t pareada, p < 0.001).

Código y resultados: [github.com/AshlyChavez/adaptauth-ml](https://github.com/AshlyChavez/adaptauth-ml)